# CATSS-based Greek correction pipeline

Corrects `hebrew_words[*].greek` in the per-chapter bible JSONs using the
CCAT/CATSS parallel Hebrew–Greek database (as rendered `.par.html` files in
`~/m_penn/out1/`).

**Two modes** controlled by `APPLY_CATSS_CHANGES`:
- `0` → report only. Counts what would change, prints verse-level diffs.
- `1` → also write back the condensed JSON, using the same compact
  serializer as `4correct_json.ipynb`.

This notebook only touches **Greek** (`hebrew_words[*].greek` and
`leftover_greek`). It never touches Hebrew or Latvian — those are owned
by `4correct_json.ipynb` and should be run first.


## config

In [ ]:
APPLY_CATSS_CHANGES = 1   # 0 = report only, 1 = write back
verbose = 1               # per-verse diff printing
verbose_skipped = 1       # print verses where we couldn't make any fix
PAR_DIR = "../m_penn/ccat_archive/gopher/text/religion/biblical/parallel"  # folder with raw .par files (e.g. 10_Ruth.par, 20.Psalms.par)
BIBLE_DIR = "bible"        # folder with per-book per-chapter json files

SYNC_LEFTOVERS_FROM_CATSS_INTO_JSON = 0 #for each verse call DO NOT or DO sync leftovers (catt -> json, json remove)
# When ON, any word currently in a JSON verse's leftover_greek whose
# normalized key does NOT appear anywhere in that verse's CATSS content
# (neither mapped nor leftover) is removed. This cleans up the residual
# "wrong-verse leftovers" that the previous AI pipeline created in books
# where LXX and MT use different verse numbering (1 Samuel 24, Daniel 3-4,
# Numbers 17, Psalms titles, etc.). The correct verse will get those words
# from its own CATSS pass.
CATSS_AUTHORITATIVE_LEFTOVERS = 1

# For duplicated CATSS books (JoshuaA/B, JudgesA/B, DanielOG/Th) we run ALL
# of them — later passes only apply safe fixes on top of the earlier pass,
# so no information is lost and any extra agreement from the second tradition
# just gives more chances to correct AI-made mappings.


## imports + CATSS → bible book name mapping

In [ ]:
import os, re, json, html as html_mod
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.notebook import tqdm

PAR_DIR_P = Path(os.path.expanduser(PAR_DIR))

# CATSS filename stem  →  bible/ folder name
# Books with no corresponding MT Hebrew folder (Ps151, 1Esdras, Sirach, Baruch)
# are mapped to None and skipped.
CATSS_TO_BIBLE = {
    "01.Genesis":   "genesis",
    "02.Exodus":    "exodus",
    "03.Lev":       "leviticus",
    "04.Num":       "numbers",
    "05.Deut":      "deuteronomy",
    "06.JoshB":     "joshua",
    "07.JoshA":     "joshua",
    "08.JudgesB":   "judges",
    "09.JudgesA":   "judges",
    "10.Ruth":      "ruth",
    "11.1Sam":      "1_samuel",
    "12.2Sam":      "2_samuel",
    "13.1Kings":    "1_kings",
    "14.2Kings":    "2_kings",
    "15.1Chron":    "1_chronicles",
    "16.2Chron":    "2_chronicles",
    "17.1Esdras":   None,   # deuterocanonical, no MT
    "18.Esther":    "esther",
    "18.Ezra":      "ezra",
    "19.Neh":       "nehemiah",
    "20.Psalms":    "psalms",
    "22.Ps151":     None,   # deuterocanonical
    "23.Prov":      "proverbs",
    "24.Qoh":       "ecclesiastes",
    "25.Cant":      "songs",
    "26.Job":       "job",
    "27.Sirach":    None,   # deuterocanonical
    "28.Hosea":     "hosea",
    "29.Micah":     "micah",
    "30.Amos":      "amos",
    "31.Joel":      "joel",
    "32.Jonah":     "jonah",
    "33.Obadiah":   "obadiah",
    "34.Nahum":     "nahum",
    "35.Hab":       "habakkuk",
    "36.Zeph":      "zephaniah",
    "37.Haggai":    "haggai",
    "38.Zech":      "zechariah",
    "39.Malachi":   "malachi",
    "40.Isaiah":    "isaiah",
    "41.Jer":       "jeremiah",
    "42.Baruch":    None,   # deuterocanonical
    "43.Lam":       "lamentations",
    "44.Ezekiel":   "ezekiel",
    "45.DanielOG":  "daniel",
    "46.DanielTh":  "daniel",
}

# Inverse: bible folder → list of CATSS stems (in the order we want to apply)
# DanielTh first (matches MT chapter structure more closely), then DanielOG
# as a second-opinion pass. JoshB/JudgesB first (LXX mainstream), then A.
BIBLE_TO_CATSS = defaultdict(list)
_order_hint = {
    "joshua": ["06.JoshB", "07.JoshA"],
    "judges": ["08.JudgesB", "09.JudgesA"],
    "daniel": ["46.DanielTh", "45.DanielOG"],
}
for stem, bible in CATSS_TO_BIBLE.items():
    if bible is None:
        continue
    BIBLE_TO_CATSS[bible].append(stem)
for bible, ordered in _order_hint.items():
    if bible in BIBLE_TO_CATSS:
        BIBLE_TO_CATSS[bible] = ordered

print(f"PAR_DIR resolved to: {PAR_DIR_P}")
print(f"{len(BIBLE_TO_CATSS)} bible folders covered by CATSS")


## Verse-number remapping (MT JSON ↔ CATSS)

CATSS `.par.html` files use **English/LXX** chapter-verse numbering for
several books (1 Samuel 23/24 boundary, Numbers 17, Daniel 3–4, Psalms
titles, etc.), while our JSONs use **MT** numbering.

The mapping is **verbatim copied** from cell 7 of `4correct_json.ipynb`
(`l65_to_hb` / `l65_to_hb_merge`) because the Latvian 1965 Bible uses the
same English verse boundaries that CATSS uses. If you update the table in
`4correct_json.ipynb`, also update it here.

Merge/split cases (marked with `"!"` placeholder in the original derivation)
are handled as **skip** in this notebook — we don't try to split CATSS cells
across a JSON merged verse, because that would require fragile text
alignment. Those verses are reported and left untouched.


In [ ]:
#split strings ir pirmaaas daljas nobeigums, ieskaitot, naakamaa dalja ir peec taa ko noraada
# l65_to_hb_merge[(('1_chronicles', 12, 4), "trīsdesmit")] = (('1_chronicles', 12, 4), ('1_chronicles', 12, 5))
# pirmaa dala ietvers sevii visu liidz vaardam triisdesmit ieskaitot (12,4), otraa tuuliit aiz taa triisdesmit(12,5)
# lv vietturis : ebreju vietturis
l65_to_hb_merge = {}
#hb_to_l65!!!
t={
('1_chronicles', 5, 27) : ('1_chronicles', 6, 1),
('1_chronicles', 5, 28) : ('1_chronicles', 6, 2),
('1_chronicles', 5, 29) : ('1_chronicles', 6, 3),
('1_chronicles', 5, 30) : ('1_chronicles', 6, 4),
('1_chronicles', 5, 31) : ('1_chronicles', 6, 5),
('1_chronicles', 5, 32) : ('1_chronicles', 6, 6),
('1_chronicles', 5, 33) : ('1_chronicles', 6, 7),
('1_chronicles', 5, 34) : ('1_chronicles', 6, 8),
('1_chronicles', 5, 35) : ('1_chronicles', 6, 9),
('1_chronicles', 5, 36) : ('1_chronicles', 6, 10),
('1_chronicles', 5, 37) : ('1_chronicles', 6, 11),
('1_chronicles', 5, 38) : ('1_chronicles', 6, 12),
('1_chronicles', 5, 39) : ('1_chronicles', 6, 13),
('1_chronicles', 5, 40) : ('1_chronicles', 6, 14),
('1_chronicles', 5, 41) : ('1_chronicles', 6, 15)
}
for i in range(1, 66+1):
    t[('1_chronicles', 6, i)]=('1_chronicles', 6, i+15)
l65_to_hb={}
for k, v in t.items():
    l65_to_hb[v]=k

###########################################################eo REVERSE!!!!##########################################
# CATSS==WEST/LEN
# l65_to_hb_merge[(('1_chronicles', 12, 4), "trīsdesmit")] = (('1_chronicles', 12, 4), ('1_chronicles', 12, 5))
# for i in range(5, 39+1):
#     l65_to_hb[('1_chronicles', 12, i)] = ('1_chronicles', 12, i+1)
l65_to_hb_merge [(('1_chronicles', 12, 40), ('1_chronicles', 12, 41))] = ('1_chronicles', 12, 40)


for l, h in zip(range(1, 14+1), range(21, 34+1)):
    l65_to_hb[('1_kings', 5, l)]=('1_kings', 4, h)

for i in range(15, 32+1):
    l65_to_hb[('1_kings', 5, i)]=('1_kings', 5, i-14)

#merge 43 + 44
l65_to_hb_merge[(('1_kings', 22, 43), ('1_kings', 22, 44))]= ('1_kings', 22, 43)

#now update mapping
for i in range(43, 53+1):
    l65_to_hb[('1_kings', 22, i+1)]=('1_kings', 22, i)

#1 samuel 20:42+
# ======================= CATSS==WEST/LEN
#l65_to_hb[('1_samuel', 20, 43)]=('1_samuel', 21, 1)
#for i in range(1, 13+1):
#    l65_to_hb[('1_samuel', 21, i)]=('1_samuel', 21, i+1)
#14, 15 merged => 14 hebrw
#l65_to_hb_merge[(('1_samuel', 21, 14), ('1_samuel', 21, 15))] = ('1_samuel', 21, 15)

#1 samuel 23:28+
t={}
l65_to_hb[('1_samuel', 24, 1)] = ('1_samuel', 23, 29)
for i in range(2, 23+1):
    l65_to_hb[('1_samuel', 24, i)]=('1_samuel', 24, i-1)
# for k, v in t:
#     l65_to_hb[v] = k

#2chronicles 2:1
l65_to_hb[('2_chronicles', 1, 18)] = ('2_chronicles', 2, 1)
for i in range(1, 17+1):
    l65_to_hb[('2_chronicles', 2, i)]=('2_chronicles', 2, i+1)

#2chronicles 14:1
l65_to_hb[('2_chronicles', 13, 23)] = ('2_chronicles', 14, 1)
for i in range(1, 14+1):
    l65_to_hb[('2_chronicles', 14, i)]=('2_chronicles', 14, i+1)

#2kings 12:1-21
l65_to_hb[('2_kings', 12, 1)] = ('2_kings', 11, 21)
for i in range(2, 22+1):
    l65_to_hb[('2_kings', 12, i)]=('2_kings', 12, i-1)

#2 sam 18:33
l65_to_hb[('2_samuel', 19, 1)] = ('2_samuel', 18, 33)
for i in range(2, 44+1):
    l65_to_hb[('2_samuel', 19, i)]=('2_samuel', 19, i-1)

#daniel 3 OG
l65_to_hb[('daniel', 3, 31)] = ('daniel', 4, 1)
l65_to_hb[('daniel', 3, 32)] = ('daniel', 4, 2)
l65_to_hb[('daniel', 3, 33)] = ('daniel', 4, 3)
#obsolote, now merge ir rghtfully updating the map.l65_to_hb[('daniel', 3, 32)] = ('daniel', 4, 2), 3

for i in range(1, 34+1):
    l65_to_hb[('daniel', 4, i)] = ('daniel', 4, i+3)

#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!1 O T R Ā D I !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# 1 lv verse = 2x heb verses, TODO: SPLIT THEM ON WORD BEING LAST IN 1st part
# 5:31 not exists in OG
#l65_to_hb_merge[(('daniel', 5, 30), "nogalināts")] = (('daniel', 5, 30), ('daniel', 5, 31))
#~~~~~~~~~~~~~~~
l65_to_hb[('deuteronomy', 13, 1)] = ('deuteronomy', 12, 32)
for i in range(2, 19+1):
    l65_to_hb[('deuteronomy', 13, i)]=('deuteronomy', 13, i-1)

l65_to_hb[('deuteronomy', 23, 1)] = ('deuteronomy', 22, 30)
for i in range(2, 26+1):
    l65_to_hb[('deuteronomy', 23, i)]=('deuteronomy', 23, i-1)

l65_to_hb[('deuteronomy', 28, 69)] = ('deuteronomy', 29, 1)
l65_to_hb_merge[(('deuteronomy', 29, 1), ('deuteronomy', 29, 2))]=('deuteronomy', 29, 2)

#ecclesiates
l65_to_hb[('ecclesiastes', 4, 17)] = ('ecclesiastes', 5, 1)
for i in range(1, 19+1):
    l65_to_hb[('ecclesiastes', 5, i)]=('ecclesiastes', 5, i+1)

l65_to_hb[('exodus', 1, 21)] = ('exodus', 1, 22)

l65_to_hb[('exodus', 7, 26)] = ('exodus', 8, 1)
l65_to_hb[('exodus', 7, 27)] = ('exodus', 8, 2)
l65_to_hb[('exodus', 7, 28)] = ('exodus', 8, 3)
l65_to_hb[('exodus', 7, 29)] = ('exodus', 8, 4)
for i in range(1, 28+1):
    l65_to_hb[('exodus', 8, i)]=('exodus', 8, i+4)

l65_to_hb[('exodus', 21, 37)] = ('exodus', 22, 1)
for i in range(1, 30+1):
    l65_to_hb[('exodus', 22, i)]=('exodus', 22, i+1)
#ezekiel
for i in range(45, 49+1):
    l65_to_hb[('ezekiel', 21, i-44)]=('ezekiel', 20, i)
for i in range(6, 37+1):
    l65_to_hb[('ezekiel', 21, i)]=('ezekiel', 21, i-5)

# genesis 3 panti savienoti vienā un izlaista 1 tauta
l65_to_hb_merge[(('genesis', 15, 19), "kadmoniešus", "refajiešus")] = (('genesis', 15, 19), ('genesis', 15, 20), ('genesis', 15, 21))

l65_to_hb_merge[(('genesis', 24, 65),('genesis', 24, 66))] = ('genesis', 24, 65)
for i in range(67, 68+1):
    l65_to_hb[('genesis', 24, i)]=('genesis', 24, i-1)

l65_to_hb[('genesis', 32, 1)]=('genesis', 31, 55)
for i in range(2, 33+1):
    l65_to_hb[('genesis', 32, i)]=('genesis', 32, i-1)

#hosea 1
l65_to_hb[('hosea', 2, 1)] = ('hosea', 1, 10)
l65_to_hb[('hosea', 2, 2)] = ('hosea', 1, 11)
for i in range(3, 25+1):
    l65_to_hb[('hosea', 2, i)]=('hosea', 2, i-2)

l65_to_hb[('hosea', 12, 1)] = ('hosea', 11, 12)
for i in range(2, 15+1):
    l65_to_hb[('hosea', 12, i)]=('hosea', 12, i-1)

l65_to_hb[('hosea', 14, 1)] =('hosea', 13, 16)
for i in range(2, 10+1):
    l65_to_hb[('hosea', 14, i)]=('hosea', 14, i-1)

#isaiah
l65_to_hb[('isaiah', 8, 23)] = ('isaiah', 9, 1)
for i in range(1, 20+1):
    l65_to_hb[('isaiah', 9, i)]=('isaiah', 9, i+1)

l65_to_hb[('jeremiah', 8, 23)] = ('jeremiah', 9, 1)
for i in range(1, 25+1):
    l65_to_hb[('jeremiah', 9, i)]=('jeremiah', 9, i+1)

# job
for i in range(25, 32+1):
    l65_to_hb[('job', 40, i)]=('job', 41, i-24)
for i in range(1, 26+1):
    l65_to_hb[('job', 41, i)]=('job', 41, i+8)
#joel
for i in range(1, 5+1):
    l65_to_hb[('joel', 3, i)]=('joel', 2, i+27)
for i in range(1, 21+1):
    l65_to_hb[('joel', 4, i)]=('joel', 3, i)

#jonah
l65_to_hb[('jonah', 2, 1)] = ('jonah', 1, 17)
for i in range(2, 11+1):
    l65_to_hb[('jonah', 2, i)]=('jonah', 2, i-1)

#leviticus
for i in range(20, 26+1):
    l65_to_hb[('leviticus', 5, i)]=('leviticus', 6, i-19)
for i in range(1, 23+1):
    l65_to_hb[('leviticus', 6, i)]=('leviticus', 6, i+7)

#malachi
for i in range(19, 24+1):
    l65_to_hb[('malachi', 3, i)]=('malachi', 4, i-18)
#malachi 4 in Latvian does not exist!!

#micah
l65_to_hb[('micah', 4, 14)] = ('micah', 5, 1)
for i in range(1, 14+1):
    l65_to_hb[('micah', 5, i)]=('micah', 5, i+1)

#nahum
l65_to_hb[('nahum', 2, 1)] = ('nahum', 1, 15)
for i in range(2, 14+1):
    l65_to_hb[('nahum', 2, i)]=('nahum', 2, i-1)

#nehemiah
l65_to_hb[('nehemiah', 10, 1)] = ('nehemiah', 9, 38)
for i in range(2, 40+1):
    l65_to_hb[('nehemiah', 10, i)]=('nehemiah', 10, i-1)


#numbers
for i in range(36, 50+1):
    l65_to_hb[('numbers', 17, i-35)]=('numbers', 16, i)
for i in range(16, 28+1):
    l65_to_hb[('numbers', 17, i)]=('numbers', 17, i-15)

l65_to_hb[('numbers', 30, 1)] = ('numbers', 29, 40)
for i in range(2, 17+1):
    l65_to_hb[('numbers', 30, i)]=('numbers', 30, i-1)

#psalms
hb_to_l65_splittable={}
def add_psalms_merging_verse_n_in_1965(chap, maxverse, mergverses=2):
    mergekey = []
    for i in range (1, mergverses+1):
        mergekey.append(('psalms', chap, i))
    l65_to_hb_merge[tuple(mergekey)] = ('psalms', chap, 1)
    for i in range(mergverses+1, maxverse+1):
        l65_to_hb[('psalms', chap, i)]=('psalms', chap, i-(mergverses-2)-1)
    #now remove the extra pointers::
    for i in range (mergverses-1):
        #fake verse so that key is not overwritten...
        l65_to_hb[('xpsalms', chap, maxverse+1+i)]=('psalms', chap, maxverse-i)

        
add_psalms_merging_verse_n_in_1965(3, 9)
add_psalms_merging_verse_n_in_1965(4, 9)
add_psalms_merging_verse_n_in_1965(5, 13)
add_psalms_merging_verse_n_in_1965(6, 11)
add_psalms_merging_verse_n_in_1965(7, 18)
add_psalms_merging_verse_n_in_1965(8, 10)
add_psalms_merging_verse_n_in_1965(9, 21)
#how come, but this is not like in Latvian65 exception of rule...
#add_psalms_merging_verse_1_in_1965(11, 8)
add_psalms_merging_verse_n_in_1965(12, 9)

add_psalms_merging_verse_n_in_1965(13, 5)
# verse 5 is 5 and 6 split on B/Y$W(T/K
#l65_to_hb_split = {('psalms', 13, 5): ['B/Y$W(T/K', ('psalms', 13, 5), ('psalms', 13, 6)]}
#now fix the added overwrite....
del l65_to_hb[('xpsalms', 13, 6)]
hb_to_l65_splittable [('psalms', 13, 5)] = (('psalms', 13, 6), 'B/Y$W(T/K', 0)
hb_to_l65_splittable [('psalms', 13, 6)] = (('psalms', 13, 6), 'B/Y$W(T/K', 1)
add_psalms_merging_verse_n_in_1965(18, 51)
add_psalms_merging_verse_n_in_1965(19, 15)
add_psalms_merging_verse_n_in_1965(20, 10)
add_psalms_merging_verse_n_in_1965(21, 14)
add_psalms_merging_verse_n_in_1965(22, 32)
add_psalms_merging_verse_n_in_1965(30, 13)
#so far ok, therefore will leave the rest of psalms as is no check
add_psalms_merging_verse_n_in_1965(31, 25)
add_psalms_merging_verse_n_in_1965(34, 23)
add_psalms_merging_verse_n_in_1965(36, 13)
add_psalms_merging_verse_n_in_1965(38, 23)
add_psalms_merging_verse_n_in_1965(39, 14)
add_psalms_merging_verse_n_in_1965(40, 18)
add_psalms_merging_verse_n_in_1965(41, 14)
add_psalms_merging_verse_n_in_1965(42, 12)
add_psalms_merging_verse_n_in_1965(44, 27)
add_psalms_merging_verse_n_in_1965(45, 18)
add_psalms_merging_verse_n_in_1965(46, 12)
add_psalms_merging_verse_n_in_1965(47, 10)
add_psalms_merging_verse_n_in_1965(48, 15)
add_psalms_merging_verse_n_in_1965(49, 21)

#also exception, 3 verses not 2 in catss
add_psalms_merging_verse_n_in_1965(51, 21, 3)
#also exception, 3 verses not 2 in catss
add_psalms_merging_verse_n_in_1965(52, 11, 3)

add_psalms_merging_verse_n_in_1965(53, 7)

#also exception, 3 verses not 2 in catss
#!!!!! was 8 wrong!!! take looka @mappings ALL
add_psalms_merging_verse_n_in_1965(54, 9, 3)

add_psalms_merging_verse_n_in_1965(55, 24)
add_psalms_merging_verse_n_in_1965(56, 14)
add_psalms_merging_verse_n_in_1965(57, 12)
add_psalms_merging_verse_n_in_1965(58, 12)
add_psalms_merging_verse_n_in_1965(59, 18)
#also exception, 3 verses not 2 in catss
add_psalms_merging_verse_n_in_1965(60, 14, 3)
add_psalms_merging_verse_n_in_1965(61, 9)
add_psalms_merging_verse_n_in_1965(62, 13)
add_psalms_merging_verse_n_in_1965(63, 12)
add_psalms_merging_verse_n_in_1965(64, 11)
add_psalms_merging_verse_n_in_1965(65, 14)
add_psalms_merging_verse_n_in_1965(67, 8)
add_psalms_merging_verse_n_in_1965(68, 36)
add_psalms_merging_verse_n_in_1965(69, 37)
add_psalms_merging_verse_n_in_1965(70, 6)
add_psalms_merging_verse_n_in_1965(75, 11)
add_psalms_merging_verse_n_in_1965(76, 13)
add_psalms_merging_verse_n_in_1965(77, 21)
add_psalms_merging_verse_n_in_1965(80, 20)
add_psalms_merging_verse_n_in_1965(81, 17)
add_psalms_merging_verse_n_in_1965(83, 19)
add_psalms_merging_verse_n_in_1965(84, 13)
add_psalms_merging_verse_n_in_1965(85, 14)
add_psalms_merging_verse_n_in_1965(88, 19)
add_psalms_merging_verse_n_in_1965(89, 53)
add_psalms_merging_verse_n_in_1965(92, 16)
add_psalms_merging_verse_n_in_1965(102, 29)
add_psalms_merging_verse_n_in_1965(108, 14)
add_psalms_merging_verse_n_in_1965(140, 14)
add_psalms_merging_verse_n_in_1965(142, 8)


#song
l65_to_hb_merge[(('songs', 4, 16),('songs', 4, 17))] = ('songs', 4, 16)
l65_to_hb[('songs', 7, 1)] = ('songs', 6, 13)
for i in range(2, 14+1):
    l65_to_hb[('songs', 7, i)]=('songs', 7, i-1)

#zechariah
for i in range(18, 21+1):
    l65_to_hb[('zechariah', 2, i-17)]=('zechariah', 1, i)
for i in range(5, 17+1):
    l65_to_hb[('zechariah', 2, i)]=('zechariah', 2, i-4)
#found out..
l65_to_hb_merge[(('genesis', 14, 20),('genesis', 14, 21))] = ('genesis', 24, 20)
l65_to_hb[('genesis', 14, 22)]=('genesis', 14, 21)
l65_to_hb_merge[(('genesis', 14, 23), "zeme")]=(('genesis', 14, 22), ('genesis', 14, 23))


# ebreju vietturis : lv vietturis
hb_to_l65 = {v: k for k, v in l65_to_hb.items() if v[0]!= '!'}
hb_to_l65_merge = {v: k for k, v in l65_to_hb_merge.items()}
for k, v in l65_to_hb_merge.items():
    if(k[0][0] == 'psalms'):
        continue
    #multiple hebrews to single latvian
    if isinstance(v[1], tuple):
        for resolvedVerse in v:
            #this is case where split string is specified for single latvian verse
            if(isinstance(k[0], tuple) and isinstance(k[1], str)):
                hb_to_l65[resolvedVerse] = ("!"+k[0][0], k[0][1], k[0][2])
                l65_to_hb[k[0]] = ("!", -1, -1)
            #also never happens in current list, what was the reasoning putting else here, "just so that it is"?
            else:
                hb_to_l65[resolvedVerse] = ("!"+k[0], k[1], k[2])
                l65_to_hb[k[0]] = ("!", -1, -1)
#value is single hebrew verse. so if key 2nd is tuple not string then
    elif isinstance(k[1], tuple):
        #where to split the hebrew verse (no case actually in current list!
        if(isinstance(v[0], tuple) and isinstance(v[1], str)):
            for resolvedVerse in k:
                hb_to_l65[resolvedVerse] = ("!"+v[0][0], v[0][1], v[0][2])
                hb_to_l65[v[0]] = ("!", -2, -2)
        #single heb verse maps to multiple lvs, so put pointer mark ! (watch merges instead of direct)
        #and the rest two just placeholders so that they are not misused anywhere as misguiding references
        else:
            for resolvedVerse in k:
                l65_to_hb[resolvedVerse] = ("!"+v[0], v[1], v[2])
            #print(v[0])
            hb_to_l65[v] = ("!", -2, -2)
### false alarm, this turns out was standard case of 2 hb verses in single lv verse.
#    elif isinstance(k[1], str) and len(v) == 3:
#        #"part of the verse is one verse in hebrew". for now i will leave the latvian to hebrew map not specified, as its not used for now anyways
#        hb_to_l65[v] = ("!", -2, -2)
#        #print(v)
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! IZLAISTS PANTS 1965
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
hb_to_l65[('exodus', 1, 21)] = None


### helper: json verse key → CATSS verse key

In [ ]:
def catss_key_for_json(book, chap, verse):
    '''
    Translate a JSON (MT) verse key into the CATSS verse key.
    Returns:
      (catss_key, status)
      status: 'identity' | 'remapped' | 'skip_merge' | 'skip_missing'

    Note on relationship with remapp(): after remapp() runs, catss_map is
    keyed by JSON-MT verse coords (not raw CATSS coords) — split/merge
    cases are already resolved into the JSON's coord space. So for the
    "lookup" purpose of process_chapter, identity is what we want; this
    function exists primarily to detect skip cases (None / "!" pointers)
    and surface them as stats.
    '''
    k = (book, chap, verse)
    if k not in hb_to_l65:
        # most verses are not in the dict — they map identity (CATSS uses
        # MT numbering for these books)
        return k, 'identity'
    v = hb_to_l65[k]
    if v is None:
        # MT verse that doesn't exist in CATSS (e.g. exodus 1:21)
        return None, 'skip_missing'
    if isinstance(v[0], str) and v[0].startswith('!'):
        # placeholder marker — this MT verse is part of a merge/split
        # group whose actual mapping lives in hb_to_l65_merge or
        # hb_to_l65_splittable. For psalms the splittable case is handled
        # by remapp() and we want to NOT skip it here.
        if k in hb_to_l65_splittable:
            # remapp will split CATSS by beta-code token boundary
            return k, 'identity'
        # otherwise we can't do a clean per-word fix on this verse
        return None, 'skip_merge'
    if isinstance(v[0], str) and v[0].startswith('x'):
        # synthetic placeholder we use to keep psalms-merge entries from
        # overwriting each other ('xpsalms' keys); should not appear as a
        # real JSON verse, but if it does, treat as missing.
        return None, 'skip_missing'
    return v, 'remapped'

def remapp(par_verses, book):
    '''
    Re-key par_verses (CATSS-keyed: {(ch,vs): [cells]}) into the JSON's
    MT-verse coord space. Resolves the four cases:

      1. hb_to_l65[k] -> (book, ch, vs)        : copy cells from that CATSS verse
      2. hb_to_l65_merge[k] -> tuple of CATSS keys : concat cells from all of them
      3. hb_to_l65_splittable[k] -> (catss_key, beta_token, part_idx) :
            split CATSS verse cells on the beta-code Hebrew word boundary
      4. otherwise: pass through identity

    Defensive: skip None pointers and '!'/'x' placeholder pointers so we
    never KeyError on (-1,-1) / (-2,-2) sentinels.
    '''
    if not book:
        return par_verses
    result = {}
    if not par_verses:
        return result
    maxchap = max(ch for (ch, vs) in par_verses)
    for ch in range(1, maxchap + 1):
        chap_verses = [vs for (c, vs) in par_verses if c == ch]
        if not chap_verses:
            continue
        maxvs = max(chap_verses)
        for vs in range(1, maxvs + 1):
            jkey = (book, ch, vs)

            # ── case 1: direct identity / remap pointer ─────────────
            if jkey in hb_to_l65:
                pointer = hb_to_l65[jkey]
                # None: JSON verse has no CATSS counterpart (e.g. exodus 1:21)
                if pointer is None:
                    continue
                # 'x' placeholder: synthetic key (xpsalms ...) — not a real verse
                if isinstance(pointer[0], str) and pointer[0].startswith("x"):
                    continue
                # '!' marker: this jkey is part of a merge/split group; the
                # real mapping is in hb_to_l65_merge or hb_to_l65_splittable.
                # Fall through to those branches below instead of trying
                # par_verses[(-1,-1)] / par_verses[(-2,-2)].
                if isinstance(pointer[0], str) and pointer[0].startswith("!"):
                    pass  # fall through
                else:
                    src_key = (pointer[1], pointer[2])
                    if src_key in par_verses:
                        result[(ch, vs)] = par_verses[src_key]
                    continue

            # ── case 2: merge — multiple CATSS verses → one JSON verse ──
            if jkey in hb_to_l65_merge:
                pointer = hb_to_l65_merge[jkey]
                aggregated = []
                for i in range(len(pointer)):
                    src_key = (pointer[i][1], pointer[i][2])
                    if src_key in par_verses:
                        aggregated.extend(par_verses[src_key])
                result[(ch, vs)] = aggregated
                continue

            # ── case 3: splittable — one CATSS verse → multiple JSON verses ──
            if jkey in hb_to_l65_splittable:
                pointer = hb_to_l65_splittable[jkey]
                src_key = (pointer[0][1], pointer[0][2])
                if src_key not in par_verses:
                    continue
                original_cells = par_verses[src_key]
                split_token = pointer[1]
                part_idx = pointer[2]
                newlist = []
                if part_idx == 0:
                    # first half: cells up to AND INCLUDING the one whose
                    # heb_trans equals the split token
                    for cell in original_cells:
                        newlist.append(cell)
                        if cell.get("heb_trans") == split_token:
                            break
                else:
                    # second half: cells AFTER the split-token cell
                    saw_split = False
                    for i, cell in enumerate(original_cells):
                        if saw_split:
                            newlist.extend(original_cells[i:])
                            break
                        if cell.get("heb_trans") == split_token:
                            saw_split = True
                result[(ch, vs)] = newlist
                continue

            # ── case 4: identity fallback ──────────────────────────
            if (ch, vs) in par_verses:
                result[(ch, vs)] = par_verses[(ch, vs)]
    return result


## CATSS raw `.par` parser


Parses a whole-book raw `.par` file (CCAT parallel Hebrew–Greek) into
`{(chapter, verse): [cells]}` where each cell captures one alignment row.

Drop-in replacement for the old `.par.html` parser — produces the same
cell schema (`heb_words`, `colb_words`, `grk_words`, `is_minus`, `is_plus`,
`transpose_marker`, `transpose_target`, `per_word_greek`, `is_particle`),
so `flatten_cells` and `verse_catss_diff` keep working unchanged.

Raw .par row format (per CCAT 00.ReadReParallel spec):
- blank lines separate verses
- `Book Ch:Vs` (or `Book N` for single-chapter books) marks a verse
- data rows are TAB-separated: `HEB-side <TAB> GRK-side`
- col-b retroversions appear inline after `=` markers in the Hebrew side
- `--+` Hebrew = LXX plus (no MT counterpart)
- `---` Greek = LXX minus (no Greek for this MT word)
- ` ^` on Hebrew side with `^^^` Greek = transposition source
- `^ ^^^ =TARGET` (or `^^^ ^ =TARGET`) on Hebrew side = transposition target
- `{...}` Hebrew side = particle row (Greek is a cross-ref to a word
  already aligned elsewhere — disregarded by the indexing program;
  becomes leftover_greek in our output)
- `''` = long-minus/plus block marker (skipped)
- `*X` / `**X` = ketib / qere prefixes (stripped during transliteration)
- `?` / `??` = questionable equivalence (stripped)
- `,,a` = Aramaic-section marker (stripped)
- `.lb` `.w` `.m` `.s` `.t` `.x` etc = morphology annotations (stripped)
- `{p}` `{!}` `{d}` `{*}` `{**}` `{...XXX}` = inline meta-annotations (stripped)
- trailing `#` = continuation marker (line continues onto next line)

Hebrew tokens are transliterated Michigan-Claremont→Unicode Hebrew so
`strip_hebrew_for_match` can compare them against the JSON's Hebrew strings.
Greek tokens are transliterated TLG Beta Code→Unicode Greek so `_greek_key`
(NFD lowercase, drop combining marks) can compare them against the JSON.


In [ ]:
# ── Raw .par parser (replaces the old .par.html / BeautifulSoup parser) ──
# Produces the same cell schema flatten_cells expects: heb_words,
# colb_words, grk_words, is_minus, is_plus, is_particle,
# transpose_marker, transpose_target, per_word_greek.
# Hebrew is transliterated to Unicode Hebrew; Greek to Unicode Greek.


import re
import unicodedata
from collections import defaultdict


# ─── Michigan-Claremont → Unicode Hebrew transliteration ─────────────────────
# (consonant-only — strip_hebrew_for_match in the notebook only looks at
#  U+05D0–U+05EA so vowels/cantillation aren't needed for matching)

_HEB_CONSONANT_MAP = {
    ')': '\u05d0', 'B': '\u05d1', 'G': '\u05d2', 'D': '\u05d3',
    'H': '\u05d4', 'W': '\u05d5', 'Z': '\u05d6', 'X': '\u05d7',
    '+': '\u05d8', 'Y': '\u05d9', 'K': '\u05db', 'L': '\u05dc',
    'M': '\u05de', 'N': '\u05e0', 'S': '\u05e1', '(': '\u05e2',
    'P': '\u05e4', 'C': '\u05e6', 'Q': '\u05e7', 'R': '\u05e8',
    '#': '\u05e9', '&': '\u05e9',  '$': '\u05e9',
    'T': '\u05ea',
}


def _transliterate_heb(token):
    """Convert one Michigan-Claremont Hebrew token to Unicode Hebrew
    (consonants only, slashes preserved as ASCII '/' since
    strip_hebrew_for_match removes them)."""
    if not token:
        return ''
    # strip leading ketiv (*) / qere (**) markers
    t = token.lstrip('*')
    out = []
    for ch in t:
        if ch in _HEB_CONSONANT_MAP:
            out.append(_HEB_CONSONANT_MAP[ch])
        elif ch == '/':
            out.append('/')   # preserved; strip_hebrew_for_match drops it
        elif ch == '-':
            out.append('-')   # maqqef separator — also dropped downstream
        # vowels / cantillation digits / annotations: silently dropped
    return unicodedata.normalize('NFC', ''.join(out))


# ─── TLG Beta Code → Unicode Greek (borrowed from ccat_render.ipynb) ─────────

_GREEK_LETTERS_LOWER = {
    'A': '\u03b1', 'B': '\u03b2', 'G': '\u03b3', 'D': '\u03b4',
    'E': '\u03b5', 'Z': '\u03b6', 'H': '\u03b7', 'Q': '\u03b8',
    'I': '\u03b9', 'K': '\u03ba', 'L': '\u03bb', 'M': '\u03bc',
    'N': '\u03bd', 'C': '\u03be', 'O': '\u03bf', 'P': '\u03c0',
    'R': '\u03c1', 'S': '\u03c3', 'J': '\u03c2',
    'T': '\u03c4', 'U': '\u03c5', 'F': '\u03c6', 'X': '\u03c7',
    'Y': '\u03c8', 'W': '\u03c9', 'V': '\u03dd',
}
_GREEK_LETTERS_UPPER = {
    'A': '\u0391', 'B': '\u0392', 'G': '\u0393', 'D': '\u0394',
    'E': '\u0395', 'Z': '\u0396', 'H': '\u0397', 'Q': '\u0398',
    'I': '\u0399', 'K': '\u039a', 'L': '\u039b', 'M': '\u039c',
    'N': '\u039d', 'C': '\u039e', 'O': '\u039f', 'P': '\u03a0',
    'R': '\u03a1', 'S': '\u03a3',
    'T': '\u03a4', 'U': '\u03a5', 'F': '\u03a6', 'X': '\u03a7',
    'Y': '\u03a8', 'W': '\u03a9',
}
_GREEK_DIACRITICS = {
    ')': '\u0313', '(': '\u0314', '/': '\u0301',
    '\\': '\u0300', '=': '\u0342', '+': '\u0308', '|': '\u0345',
}


def _convert_greek_word(word):
    if not word:
        return word
    result = []
    i = 0
    n = len(word)
    while i < n:
        ch = word[i]
        if ch == '*':
            i += 1
            pending_dia = []
            while i < n and word[i] in _GREEK_DIACRITICS:
                pending_dia.append(_GREEK_DIACRITICS[word[i]])
                i += 1
            if i < n and word[i] in _GREEK_LETTERS_UPPER:
                result.append(_GREEK_LETTERS_UPPER[word[i]])
                result.extend(pending_dia)
                i += 1
            continue
        if ch in _GREEK_LETTERS_LOWER:
            letter = _GREEK_LETTERS_LOWER[ch]
            if ch == 'S':
                j = i + 1
                while j < n and word[j] in _GREEK_DIACRITICS:
                    j += 1
                if j == n:
                    letter = '\u03c2'
            result.append(letter)
            i += 1
            while i < n and word[i] in _GREEK_DIACRITICS:
                result.append(_GREEK_DIACRITICS[word[i]])
                i += 1
            continue
        if ch in _GREEK_DIACRITICS:
            result.append(_GREEK_DIACRITICS[ch])
            i += 1
            continue
        result.append(ch)
        i += 1
    return unicodedata.normalize('NFC', ''.join(result))


def _transliterate_greek_token(word):
    """Convert one TLG Beta Code token to Unicode Greek."""
    if not word or word in ('---', '--+', '^', '~', '~~~', '^^^'):
        return word
    if word.startswith('{') or word.startswith('=') or word.startswith('<'):
        return word
    if "'" in word:
        # elision apostrophe — convert each side, rejoin
        parts = word.split("'")
        return "'".join(_convert_greek_word(p) for p in parts)
    return _convert_greek_word(word)


# ─── verse header ────────────────────────────────────────────────────────────
# Matches "Ruth 1:1", "1 Sam 3:4", "Obad 4" (single-chapter), "Dan 1:1"
_VERSE_RE = re.compile(r'^([A-Za-z][A-Za-z0-9]*(?:\s+[A-Za-z][A-Za-z0-9]*)?)\s+(\d+)(?::(\d+))?\s*$')

# Standalone marker tokens to strip from word lists
_STRIP_TOKENS = {
    '?', '??', "''", "'", '---', '--', '--?', '---?',
    '^', '^^^', '~', '~~~',
    # CCAT inline annotations that sometimes appear as standalone tokens
    '{...}', '{d}', '{!}', '{p}', '{*}', '{**}',
}

# Hebrew consonant alphabet for "is this a real word" filter
_HEB_CONS = set(')BGDHWZX+YKLMNS(PCQR#&$T')

# Greek alphabet ranges (for filtering Greek tokens)
def _has_greek_letter(s):
    """True if token contains any Greek letter (beta-code maps to A-Z, but we
    can't just check ASCII because Hebrew is also ASCII-Roman). Instead,
    we accept any token that has at least one of Greek-only beta-code
    indicators: '/' '\\' '=' '+' '|' (accents/breathings — they only appear
    on Greek), OR the token is purely uppercase Roman (no consonant ambiguity
    with Hebrew). Hebrew tokens always contain at least one of: ) ( + & $ # / -
    and never accent marks. So the simple test is: does the token have an
    accent? If yes → Greek. If no but it's pure A-Z → Greek too (e.g. KAI).
    """
    if not s:
        return False
    # any Greek-only diacritic char
    if any(c in '/\\=+|' for c in s):
        # but '/' is also Hebrew morpheme separator! Disambiguate: Greek
        # diacritics come AFTER vowels (lowercase position), while Hebrew '/'
        # separates morphemes. Heuristic: if the token contains parens () or
        # uppercase Hebrew-only consonants ($ # & + ), it's Hebrew.
        if any(c in s for c in '$#&'):
            return False
        # ')' and '(' are Hebrew alef/ayin AND Greek breathing marks. In Greek
        # they always appear right after a vowel; in Hebrew they're standalone
        # consonants. We accept it as Greek if there's also an accent.
        if any(c in s for c in '/\\=+|'):
            return True
    # Pure A-Z token, no slashes/parens — could be either. The Greek side
    # never has '/' as morpheme separator, but it may have '/' as accent.
    # Easiest: caller already split rows by tab into HEB and GRK columns,
    # so this filter is mostly a sanity check. Default true for the Greek
    # side context.
    return True


def _is_hebrew_word_tok(s):
    """True if s looks like a Michigan-Claremont Hebrew word (has at least
    one Hebrew consonant char) and isn't a marker."""
    if not s or s in _STRIP_TOKENS:
        return False
    bare = s.lstrip('*')  # strip ketiv/qere prefix
    if not bare:
        return False
    # must have at least one Hebrew consonant
    return any(c in _HEB_CONS for c in bare)


def _split_heb_side(heb_raw):
    """
    Split the raw Hebrew side of a .par row into:
      (col_a_tokens, colb_tokens, flags)

    flags: dict with keys is_plus, is_minus, transpose_marker, transpose_target,
           is_particle, is_long_marker.

    Splits col-a from col-b on '=' markers. CCAT format is loose — col-b
    is everything from the first '=' to end of line; col-a is everything
    before. Inline annotations like {...XXX}, {p}, {!}, {d}, .s, .lb, ,,a
    are stripped from both columns.
    """
    flags = {
        'is_plus':          False,
        'is_minus':         False,   # set by caller from grk side
        'transpose_marker': None,
        'transpose_target': None,
        'is_particle':      False,
        'is_long_marker':   False,
    }

    s = heb_raw.strip()
    if not s:
        return [], [], flags

    if s == "''":
        flags['is_long_marker'] = True
        return [], [], flags

    if s == '--+':
        flags['is_plus'] = True
        return [], [], flags

    # particle row: heb side starts/equals "{...}" with no real word.
    # Per CCAT spec these rows are "disregarded by indexing program" — the
    # Greek word here is a cross-reference to one already aligned elsewhere
    # in the verse. We keep the row as a heb_words=[] cell so flatten_cells
    # routes the Greek to leftover_greek (matches the .par.html parser
    # behaviour, which sets is_particle=False for these rows because
    # BeautifulSoup strips '{...}' from heb_trans before the test).
    if s == '{...}':
        return [], [], flags

    # Strip inline {...} annotations BEFORE tokenizing: {...MLK}, {p}, {!},
    # {d}, {d?}, {*}, {**}, {..d}, {..r}, {..p}, {...XXX} cross-references.
    # These are all CCAT meta tokens that the .par.html parser drops.
    s_clean = re.sub(r'\{[^{}]*\}', '', s)
    # Also strip ,,a Aramaic markers and .X morphology annotations
    # that appear as separate tokens (after whitespace) — those are
    # handled per-token below.

    # Tokenize on whitespace, then walk left-to-right splitting on '='
    raw_tokens = s_clean.split()

    # Plus row may carry a col-b after --+, e.g. "--+ =W/$TY"
    if raw_tokens and raw_tokens[0] == '--+':
        flags['is_plus'] = True
        rest = raw_tokens[1:]
        colb = []
        for t in rest:
            if t.startswith('='):
                inner = t.lstrip('=').lstrip(':')
                if inner and _is_hebrew_word_tok(inner):
                    colb.append(inner)
            elif _is_hebrew_word_tok(t):
                colb.append(t)
            # else: meta marker like .lb / .w — drop
        return [], colb, flags

    # Transposition source: row has " ^" trailing on heb side and grk='^^^'
    # Transposition target: heb side is something like "^ ^^^ =TARGET" or
    # "^^^ ^ =TARGET" (both orderings appear in the raw .par files) — i.e.
    # tokens are some permutation of {'^', '^^^'} followed by one or more
    # =RETROVERSION tokens.
    has_caret = '^' in raw_tokens or '^^^' in raw_tokens

    # detect target row: tokens before the first '=' are all in {'^', '^^^'}
    # and there is at least one '=TARGET' token
    is_target_row = False
    if has_caret:
        first_eq_idx = next((i for i, t in enumerate(raw_tokens) if t.startswith('=')), -1)
        if first_eq_idx > 0 and all(t in ('^', '^^^') for t in raw_tokens[:first_eq_idx]):
            is_target_row = True

    if is_target_row:
        # extract =TARGET (the marker), AND any further =RETROVERSION tokens
        # become col-b words. The .par.html parser does this — see Ruth 1:5
        # cells 10/11 where colb shows ['ומשני'] / ['ילדיה'].
        colb = []
        saw_first_equals = False
        for t in raw_tokens[first_eq_idx:]:
            if t.startswith('='):
                inner = t.lstrip('=').lstrip(':').lstrip('@')
                if inner:
                    if not saw_first_equals:
                        flags['transpose_target'] = inner
                    if _is_hebrew_word_tok(inner):
                        colb.append(inner)
                saw_first_equals = True
            elif saw_first_equals and _is_hebrew_word_tok(t):
                colb.append(t)
        return [], colb, flags

    # transposition SOURCE: row has bare '^' but not the '^^^ ^ =TARGET' form
    if has_caret and '^' in raw_tokens:
        flags['transpose_marker'] = '^'
        # fall through to normal col-a / col-b split, but drop bare '^' / '^^^'

    # Normal split: walk tokens, anything from first '=' onwards is col-b
    col_a = []
    col_b = []
    in_colb = False
    for t in raw_tokens:
        if t == '^' or t == '^^^':
            continue
        if t.startswith('='):
            in_colb = True
            inner = t.lstrip('=').lstrip(':').lstrip('@')   # =@CBR retroversion
            if inner and _is_hebrew_word_tok(inner):
                col_b.append(inner)
            continue
        # CCAT meta-annotations (drop on either side):
        #   ,,a       Aramaic-section marker
        #   .lb .w .m .s .t .x .z .j .rd .+r etc — morphology annotations
        #   ??        questionable (also handled by _STRIP)
        if t.startswith(',,') or t.startswith('.'):
            continue
        if t in _STRIP_TOKENS:
            continue
        if in_colb:
            if _is_hebrew_word_tok(t):
                col_b.append(t)
            # else drop
        else:
            if _is_hebrew_word_tok(t):
                col_a.append(t)
            # else drop (no consonant-fallback — too risky given the
            # variety of CCAT inline meta-tokens we haven't enumerated)

    return col_a, col_b, flags


# ─── Greek side tokenization ─────────────────────────────────────────────────

# A Greek beta-code token consists of A-Z plus diacritics  / \ = + |  and
# parentheses for breathing. Anything else is structural.
_GRK_TOKEN_CHARS = set('ABGDEZHQIKLMNCOPRSJTUFXYW')  # CCAT Greek alphabet
_GRK_DIACRITICS  = set('/\\=+|()')


def _is_greek_word_tok(s):
    """Real Greek word: has at least one Greek-alphabet char and isn't
    a marker like '---' / '?' / '^^^' / '{...XXX}'."""
    if not s or s in _STRIP_TOKENS:
        return False
    if s.startswith('{') and s.endswith('}'):
        return False  # pure annotation like {d?}
    return any(c in _GRK_TOKEN_CHARS for c in s)


def _strip_inline_annotations(grk_raw):
    """
    Remove inline {...} CCAT annotations from a Greek string.
    Examples to strip:
      '{...}'                  — particle marker
      '{...TOI=S}'             — inline-content reference (but the inner
                                 word here belongs to the following cell;
                                 the .par.html parser drops these)
      '{..pEI)S}'              — preposition added in LXX (drop wrapper,
                                 keep content? .par.html drops the whole
                                 thing — so do we)
      '{d}', '{d?}', '{!}'     — pure annotations
    Strategy: drop everything inside {...} including the braces. This
    matches what .par.html rendering does (the inner refs are display-only
    cross-refs).
    """
    return re.sub(r'\{[^{}]*\}', '', grk_raw)


def _tokenize_greek_side(grk_raw):
    """
    Returns (grk_words, raw_tokens_with_minus, has_caret).
    grk_words           : real greek words only
    raw_tokens_with_minus: includes '---' tokens for inline distribution
    has_caret_tokens    : list of indices where bare '^' appeared (inline transpose)
    """
    if not grk_raw:
        return [], [], False
    s = grk_raw.strip()
    if s in ("''", "'"):
        return [], [], False
    # strip inline {...} blocks
    cleaned = _strip_inline_annotations(s)
    raw_tokens = cleaned.split()
    grk_words = []
    raw_with_minus = []
    has_caret = False
    for t in raw_tokens:
        if t in ('---', '--', '--?', '---?'):
            raw_with_minus.append('---')
            continue
        if t in ('^', '^^^'):
            has_caret = True
            continue
        if t in _STRIP_TOKENS:
            continue
        if _is_greek_word_tok(t):
            grk_words.append(t)
            raw_with_minus.append(t)
        # else: unknown junk — drop silently
    return grk_words, raw_with_minus, has_caret


# ─── Main entry point ────────────────────────────────────────────────────────

def parse_par(path):
    """
    Parse a raw CCAT .par file into {(chapter, verse): [cells]}.

    Returns the same cell shape that the old parse_par_html() emitted, so
    flatten_cells / verse_catss_diff in the notebook keep working unchanged.
    """
    raw_verses = {}   # {(ch, vs): [raw_row_dict]}
    current_key = None

    with open(path, 'r', encoding='utf-8', errors='replace') as fh:
        # First handle the line-continuation '#' marker: a line ending with
        # '#' means "continued on the next line in the OPPOSITE column".
        # CCAT spec: '# at end of line running over and at the beginning of
        # the following line in the opposite column'. We simply concatenate
        # the two — on whichever side has the '#'.
        raw_lines = [ln.rstrip('\n').rstrip('\r') for ln in fh]

    # Coalesce continuations
    coalesced = []
    i = 0
    while i < len(raw_lines):
        line = raw_lines[i]
        # crude: if line ends with '#' and there is a next non-empty line,
        # we just glue them. The parser is robust to a slightly fatter
        # line because tokenization happens per-side after split('\t').
        while line.rstrip().endswith('#') and i + 1 < len(raw_lines):
            nxt = raw_lines[i + 1]
            line = line.rstrip()[:-1] + ' ' + nxt
            i += 1
        coalesced.append(line)
        i += 1

    for line in coalesced:
        if not line.strip():
            continue
        # verse header?
        if '\t' not in line:
            m = _VERSE_RE.match(line.strip())
            if m:
                a, b = m.group(2), m.group(3)
                if b is None:
                    ch, vs = 1, int(a)
                else:
                    ch, vs = int(a), int(b)
                current_key = (ch, vs)
                if current_key not in raw_verses:
                    raw_verses[current_key] = []
            continue
        if current_key is None:
            continue

        heb_raw, _, grk_raw = line.partition('\t')

        col_a, col_b, hflags = _split_heb_side(heb_raw)
        grk_words, grk_with_minus, grk_has_caret = _tokenize_greek_side(grk_raw)

        # is_minus: greek side is purely '---' (no real greek words), and
        # heb side has real tokens (i.e. not a plus, not particle, not target)
        is_minus = (not grk_words) and ('---' in grk_with_minus) and bool(col_a)

        # CCAT inline transposition: row has real heb + real grk, and one of
        # the sides also has a '^' marker. The row itself is fine; the '^'
        # is just a marker that this cell's order differs across MT/LXX.
        # We don't model that any further — flatten_cells doesn't care.
        # But: the .par.html parser sets transpose_marker only when grk side
        # is empty/^^^. So for inline-^ rows we keep transpose_marker=None.
        if grk_words and hflags['transpose_marker'] == '^':
            hflags['transpose_marker'] = None  # this is just an inline reorder

        # per-word greek for inline --- in multi-heb-word cells
        per_word_greek = None
        if len(col_a) > 1 and '---' in grk_with_minus:
            if len(grk_with_minus) == len(col_a):
                per_word_greek = []
                for tok in grk_with_minus:
                    per_word_greek.append([] if tok == '---' else [tok])
            else:
                groups = []
                current = []
                for tok in grk_with_minus:
                    if tok == '---':
                        groups.append(current)
                        current = []
                    else:
                        current.append(tok)
                groups.append(current)
                if len(groups) == len(col_a):
                    per_word_greek = groups

        if hflags['is_long_marker']:
            # skip ''-only rows entirely (matches .par.html behaviour)
            continue

        # Transliterate to Unicode Hebrew so strip_hebrew_for_match
        # downstream can compare against the JSON's Hebrew strings.
        col_a_unicode = [_transliterate_heb(w) for w in col_a]
        col_b_unicode = [_transliterate_heb(w) for w in col_b]

        # Transliterate Greek tokens from beta-code to Unicode Greek so the
        # diff engine's _greek_key (NFD + lowercase + strip combining marks)
        # can compare against the JSON's Greek strings.
        grk_words_unicode = [_transliterate_greek_token(w) for w in grk_words]

        # per_word_greek too (for inline ---  multi-word distribution)
        per_word_greek_unicode = None
        if per_word_greek is not None:
            per_word_greek_unicode = [
                [_transliterate_greek_token(w) for w in grp]
                for grp in per_word_greek
            ]

        cell = {
            'heb_native':       ' '.join(col_a_unicode) if col_a_unicode else heb_raw.strip(),
            'heb_trans':        heb_raw.strip(),
            'heb_words':        col_a_unicode,
            'colb_words':       col_b_unicode,
            'grk_words':        grk_words_unicode,
            'per_word_greek':   per_word_greek_unicode,
            'is_minus':         is_minus,
            'is_plus':          hflags['is_plus'],
            'is_particle':      hflags['is_particle'],
            'transpose_marker': hflags['transpose_marker'],
            'transpose_target': hflags['transpose_target'],
        }
        raw_verses[current_key].append(cell)

    # ── Pass 1: resolve ^ transpositions (bidirectional, like .par.html parser)
    for key, cells in raw_verses.items():
        for i, c in enumerate(cells):
            if c['transpose_target'] is None:
                continue
            tgt = c['transpose_target']
            found = False
            for j in range(i - 1, -1, -1):
                cj = cells[j]
                if cj['transpose_marker'] != '^':
                    continue
                # source row's heb_trans first token (without ' ^') must equal target
                first_tok = cj['heb_trans'].split()[0] if cj['heb_trans'] else ''
                if first_tok == tgt:
                    cj['grk_words'] = cj['grk_words'] + c['grk_words']
                    c['grk_words'] = []
                    found = True
                    break
            if found:
                continue
            for j in range(i + 1, len(cells)):
                cj = cells[j]
                if cj['transpose_marker'] != '^':
                    continue
                first_tok = cj['heb_trans'].split()[0] if cj['heb_trans'] else ''
                if first_tok == tgt:
                    cj['grk_words'] = cj['grk_words'] + c['grk_words']
                    c['grk_words'] = []
                    break

    # ── Pass 2: legacy particle-merge.
    # Currently `is_particle` is never True (we treat {...} rows as orphan
    # heb=[] rows so their Greek routes to leftover_greek via flatten_cells,
    # matching the spec: "Equivalent reflected elsewhere in the text,
    # disregarded by indexing program"). Kept as no-op for completeness.
    final_verses = {}
    for key, cells in raw_verses.items():
        out = []
        pending = []
        for c in cells:
            if c['is_particle']:
                pending.extend(c['grk_words'])
                continue
            if pending and c['heb_words']:
                c['grk_words'] = pending + c['grk_words']
                pending = []
            out.append(c)
        if pending and out:
            out[-1]['grk_words'] = out[-1]['grk_words'] + pending
        final_verses[key] = out

    return final_verses




## Flatten CATSS cells → per-Hebrew-word expected greek

Each CATSS cell can contain 1..N Hebrew words with a single Greek
group. We flatten such that the first Hebrew word of a multi-word cell
carries the entire Greek group, and the rest carry `[]` — this matches how
our JSON encodes alignments.

Plus cells (`--+`) and any orphan particle greek get collected into
`expected_leftover_greek` for the verse.


In [ ]:
def flatten_cells(cells):
    '''
    Returns:
      flat      : list of dicts {'heb': str, 'heb_alt': [str,...], 'greek': [str,...],
                                  'group_size': int, 'group_first': bool}
                  one per Hebrew word (slash kept intact).
                  - 'heb' is the column-A (MT) form.
                  - 'heb_alt' is the column-B (LXX-parent retroversion) form(s)
                    if any, used as a Hebrew matching fallback.
                  - 'group_size' / 'group_first' as before.
                  - 'greek' is the FULL greek group on the first heb word
                    (or the per-word slice if inline --- was set).
      leftover  : list of str — greek words with no Hebrew anchor
    '''
    flat = []
    leftover = []
    for c in cells:
        if c['is_plus']:
            leftover.extend(c['grk_words'])
            continue
        if not c['heb_words']:
            if c['grk_words']:
                leftover.extend(c['grk_words'])
            continue
        gsize = len(c['heb_words'])
        colb = c.get('colb_words') or []
        # Pair colb words with heb_words positionally if counts match;
        # otherwise share the full colb list as a generic fallback for each.
        def colb_for(idx):
            if colb and len(colb) == gsize:
                return [colb[idx]]
            return list(colb)
        if c.get('per_word_greek') is not None:
            for idx, (hw, gg) in enumerate(zip(c['heb_words'], c['per_word_greek'])):
                flat.append({
                    'heb': hw, 'heb_alt': colb_for(idx), 'greek': list(gg),
                    'group_size': gsize, 'group_first': (idx == 0),
                    'group_inline_split': True,
                })
        else:
            for idx, hw in enumerate(c['heb_words']):
                flat.append({
                    'heb':   hw,
                    'heb_alt': colb_for(idx),
                    'greek': list(c['grk_words']) if idx == 0 else [],
                    'group_size': gsize, 'group_first': (idx == 0),
                    'group_inline_split': False,
                })
    return flat, leftover


## Hebrew normalization for matching

In [ ]:
# Re-defined here so the notebook is self-contained vs 4correct_json.ipynb

def strip_hebrew_for_match(s):
    '''Lowercase-ish Hebrew comparison key: consonants only, no slash,
    collapse final letters to their non-final equivalents so ם==מ etc.'''
    out = []
    FINAL_MAP = {
        '\u05DA': '\u05DB',  # ך → כ
        '\u05DD': '\u05DE',  # ם → מ
        '\u05DF': '\u05E0',  # ן → נ
        '\u05E3': '\u05E4',  # ף → פ
        '\u05E5': '\u05E6',  # ץ → צ
    }
    for c in s:
        if '\u05D0' <= c <= '\u05EA':
            out.append(FINAL_MAP.get(c, c))
    return ''.join(out).replace('/', '')

assert strip_hebrew_for_match('וַיְדַבֵּ֨ר') == strip_hebrew_for_match('ו/ידבר')
assert strip_hebrew_for_match('אֱלֹהִים֒') == strip_hebrew_for_match(strip_hebrew_for_match('אלהים'))
print('hebrew match normalizer ok')


## Diff / fix engine — per verse

Two strategies:

1. **Exact length match** (fast path): flat CATSS length == JSON `hebrew_words`
   length, AND each Hebrew key matches positionally. → replace greek lists
   wholesale, reconcile leftovers.
2. **Per-unique-word safe fixes** (fallback for mismatches): for each distinct
   Hebrew consonant key:
     - CATSS count == JSON count == 1 → safe direct fix.
     - CATSS count == JSON count == N≥2 AND all CATSS greek groups for that
       key are identical → safe bulk fix.
     - otherwise skip (ambiguous position).
   Leftover reconciliation still runs.


In [ ]:
import unicodedata as _ud

def raccs_common(text):
    '''Strip miscellaneous punctuation/quote characters from text before
    comparison. Currently only U+0027 APOSTROPHE is active — used by the
    HTML renderer as a Greek elision mark (e.g. ἀλλ' from beta code ALL').
    Per CATSS Beta Code spec there is NO apostrophe character at all
    (smooth breathing is `)` and rough breathing is `(`), so the elision
    apostrophe is purely an editor's addition. Stripping it lets words
    that differ only in this editor-added mark compare as identical, and
    the JSON's spelling is preserved by `_merge_greek_preserve_case`.

    All other entries are commented out for now; uncomment selectively
    if/when the dataset proves they should also be normalized away.
    '''
    d = {
        #ord('\\N{COMBINING ACUTE ACCENT}'):None,
        ord('\''): None,    # U+0027 APOSTROPHE  ← ACTIVE (editor's elision mark)
        #ord('’'): None,  # U+2019 RIGHT SINGLE QUOTATION MARK
        #ord('‘'): None,  # U+2018 LEFT SINGLE QUOTATION MARK
        #ord('“'): None,  # U+201C LEFT DOUBLE QUOTATION MARK
        #ord('”'): None,  # U+201D RIGHT DOUBLE QUOTATION MARK
        #ord('['): None,
        #ord(']'): None,
        #ord('-'): None,        # HYPHEN-MINUS
        #ord('⧼'): None,   # ⧼ LEFT WHITE ANGLE BRACKET
        #ord('⧽'): None,   # ⧽ RIGHT WHITE ANGLE BRACKET
        #ord('*'): None,
        #ord('⇔'): None,   # ⇔ LEFT RIGHT DOUBLE ARROW
        #ord('〉'): None,   # 〉
        #ord('〈'): None,   # 〈
        #ord('‿'): None,   # ‿ LOW LINE
        #ord('«'): None,   # «
        #ord('»'): None,   # »
        #ord('‹'): None,   # ‹
        #ord('›'): None,   # ›
        #ord('('): None,
        #ord(')'): None,
        #ord(';'): None,
        #ord(','): None,
        #ord('!'): None,
        #ord('?'): None,
        #ord('.'): None,
    }
    return _ud.normalize('NFD', text).translate(d)

def _greek_key(w):
    '''Normalized key for equality checks: lowercased, NFD-normalized,
    combining marks (accents/breathings/iota subscript) stripped, and
    `raccs_common` applied to drop the elision apostrophe (U+0027).
    Two words producing the same key are considered "the same word" and
    the JSON's spelling is preserved by the merge step.'''
    s = raccs_common(w.lower())
    return ''.join(c for c in s if _ud.category(c) != 'Mn')

def _greek_lists_equal(a, b):
    '''Compare greek lists ignoring case, accents, and the elision
    apostrophe (U+0027).'''
    if len(a) != len(b):
        return False
    return all(_greek_key(x) == _greek_key(y) for x, y in zip(a, b))

def _merge_greek_preserve_case(new_from_catss, old_from_json):
    '''
    Produce the new greek list taking CATSS's composition/order,
    but reusing the JSON's original capitalized form whenever the
    normalized key matches something the JSON already had.
    This keeps proper-noun capitalization like 'Ισραηλ', 'Μωυσῆν'.
    '''
    # index old json words by normalized key; preserve first occurrence order
    old_by_key = {}
    for w in old_from_json:
        k = _greek_key(w)
        old_by_key.setdefault(k, w)
    out = []
    for w in new_from_catss:
        k = _greek_key(w)
        out.append(old_by_key.get(k, w))
    return out


def verse_catss_diff(json_verse, catss_cells, key, verbose=True):
    '''
    Compare JSON verse against CATSS cells for one verse.
    Returns dict with:
      status          : 'exact' | 'partial' | 'length_mismatch_skip' | 'no_catss' | 'no_change'
      replaces        : list of (word_idx, old_greek, new_greek)
      leftover_adds   : list[str]
      leftover_removes: list[str]
      new_hebrew_words: list (updated)  — only if any change
      new_leftover    : list[str]       — only if any change
    '''
    global SYNC_LEFTOVERS_FROM_CATSS_INTO_JSON
    result = {
        'status': 'no_catss',
        'replaces': [],
        'leftover_adds': [],
        'leftover_removes': [],
        'new_hebrew_words': None,
        'new_leftover': None,
    }
    if catss_cells is None:
        return result

    flat, catss_leftover = flatten_cells(catss_cells)
    hebrew_words = [dict(w) for w in json_verse.get('hebrew_words', [])]
    leftover_greek = list(json_verse.get('leftover_greek', []))

    # Cleanup: prior buggy runs may have written the literal "''" CATSS
    # long-marker token into greek lists or leftover_greek. Strip those out
    # so the current run can overwrite them cleanly.
    for hw in hebrew_words:
        if hw.get('greek'):
            hw['greek'] = [g for g in hw['greek'] if g != "''"]
    leftover_greek = [g for g in leftover_greek if g != "''"]

    # Precompute the full set of Greek keys CATSS knows for this verse —
    # used by reconcile_leftover when CATSS_AUTHORITATIVE_LEFTOVERS is on.
    catss_all_greek_keys = set()
    for fl in flat:
        for g in fl['greek']:
            catss_all_greek_keys.add(_greek_key(g))
    for g in catss_leftover:
        catss_all_greek_keys.add(_greek_key(g))

    # Detect "long minus" verses: CATSS has hebrew rows but ALL of them
    # have empty greek (the entire stretch is a minus block in the LXX-parent
    # text, often marked with '' in the source .par). In such verses, CATSS
    # has nothing to teach us — the JSON's AI mappings (which probably came
    # from Rahlfs/Göttingen and DO contain the verse) are more authoritative.
    if flat and not catss_leftover and all(not fl['greek'] for fl in flat):
        result['status'] = 'catss_long_minus'
        return result

    # Build a pool of all existing json greek words (mapped + leftover) so
    # we can reuse their original casing for any CATSS word we write back.
    json_full_pool = []
    for hw in hebrew_words:
        json_full_pool.extend(hw.get('greek', []))
    json_full_pool.extend(leftover_greek)

    # Pre-compute the set of Greek keys present in the JSON's verse pool.
    # "Never invent words" rule: the engine must never write a Greek word
    # into a JSON entry unless that exact word (by _greek_key) already
    # exists somewhere in this pool. CATSS's role is to re-align existing
    # vocabulary, not to replace word forms or introduce new ones.
    json_pool_keys = {_greek_key(g) for g in json_full_pool if g}

    def _filter_to_pool(new_g):
        '''Drop any proposed greek word whose key isn't in json_pool_keys.'''
        return [g for g in new_g if _greek_key(g) in json_pool_keys]

    # ---------- Strategy 1: exact length (with paragraph-marker tolerance) ----------
    PARAGRAPH_MARKERS = {'פ', 'ס'}
    # Paragraph markers (פ/ס) appear as JSON "hebrew_words" entries but are
    # not in CATSS. We build a list of indices of real (non-marker) JSON
    # positions, so length comparison and positional walking skip them.
    real_json_indices = [i for i, hw in enumerate(hebrew_words)
                         if hw.get('hebrew', '').strip() not in PARAGRAPH_MARKERS]
    json_effective_len = len(real_json_indices)

    if len(flat) == json_effective_len:
        mismatch = False
        for fi in range(json_effective_len):
            fl = flat[fi]
            ji = real_json_indices[fi]
            hw = hebrew_words[ji]
            json_key = strip_hebrew_for_match(hw.get('hebrew', ''))
            if strip_hebrew_for_match(fl['heb']) == json_key:
                continue
            if any(strip_hebrew_for_match(alt) == json_key for alt in fl.get('heb_alt', [])):
                continue
            mismatch = True
            break
        if not mismatch:
            fi = 0
            while fi < json_effective_len:
                fl = flat[fi]
                ji = real_json_indices[fi]
                hw = hebrew_words[ji]
                gsize = fl.get('group_size', 1)
                inline_split = fl.get('group_inline_split', False)

                if gsize > 1 and not inline_split:
                    # Multi-heb-word CATSS cell with a single greek group.
                    # Collect the JSON slice across the group's real positions.
                    group_real_idx = [real_json_indices[fi + k] for k in range(gsize)]
                    catss_group_greek = list(fl['greek'])
                    json_slice = [list(hebrew_words[j].get('greek', [])) for j in group_real_idx]
                    json_concat = [g for per in json_slice for g in per]

                    if _greek_lists_equal(catss_group_greek, json_concat):
                        # JSON already has correct content; possibly with a
                        # better-than-CATSS distribution. Don't touch it.
                        fi += gsize
                        continue

                    # CATSS minus protection
                    if not catss_group_greek and json_concat:
                        fi += gsize
                        continue

                    # Morphology protection: if any JSON slice word is NOT
                    # in CATSS's vocabulary for the verse, it's a morphology
                    # variant from a different edition — don't touch this
                    # group at all.
                    slice_has_foreign = any(
                        _greek_key(g) not in catss_all_greek_keys
                        for per in json_slice for g in per
                    )
                    if slice_has_foreign:
                        fi += gsize
                        continue

                    # "Never invent words" filter applied to the catss group.
                    filtered_group = _filter_to_pool(catss_group_greek)

                    # If the filter dropped everything CATSS wanted (e.g. a
                    # morphology-only difference), there's no safe fix — the
                    # engine is not allowed to introduce new forms. Leave the
                    # JSON alone.
                    if not filtered_group and catss_group_greek:
                        fi += gsize
                        continue

                    # Try to PRESERVE the JSON's per-word distribution when
                    # possible: for each heb position, keep its existing
                    # greek words that belong to the filtered group; then
                    # fill in any missing filtered-group words on whichever
                    # position doesn't have them yet.
                    remaining = list(filtered_group)  # words still to place
                    new_per = [[] for _ in range(gsize)]

                    # First pass: keep existing JSON assignments that match
                    # a word in the filtered group (case-insensitively).
                    for k in range(gsize):
                        for gw in json_slice[k]:
                            gk = _greek_key(gw)
                            match_idx = next((ri for ri, rw in enumerate(remaining)
                                              if _greek_key(rw) == gk), None)
                            if match_idx is not None:
                                # keep it on this position, preserving JSON's casing
                                new_per[k].append(gw)
                                remaining.pop(match_idx)

                    # Second pass: place any remaining (CATSS-wanted but not
                    # yet placed) words. Prefer the first heb position as a
                    # "lump target" to match old behavior for leftover pulls.
                    if remaining:
                        # preserve JSON casing for pulls from the pool
                        pulled = _merge_greek_preserve_case(remaining, json_full_pool)
                        new_per[0].extend(pulled)

                    # Write back, recording replaces per position
                    for k in range(gsize):
                        old_g_k = json_slice[k]
                        new_g_k = new_per[k]
                        if not _greek_lists_equal(old_g_k, new_g_k):
                            result['replaces'].append((group_real_idx[k], old_g_k, new_g_k))
                            hebrew_words[group_real_idx[k]]['greek'] = new_g_k
                    fi += gsize
                    continue

                # single-word cell (or inline-split multi-word)
                old_g = list(hw.get('greek', []))
                # "Never invent words": filter CATSS proposal to json pool
                proposed = _filter_to_pool(list(fl['greek']))
                # If filter killed all CATSS content but CATSS was non-empty,
                # treat like a morphology difference — leave JSON alone.
                if not proposed and fl['greek']:
                    fi += 1
                    continue
                new_g = _merge_greek_preserve_case(proposed, json_full_pool)
                # CATSS minus protection
                if not new_g and old_g:
                    fi += 1
                    continue
                # Morphology protection: if old_g contains words that are
                # NOT in CATSS's vocabulary for this verse, they're from a
                # different Greek edition (e.g. πατὴρ vs CATSS πατρὸς).
                # Don't replace them — CATSS re-aligns, it doesn't swap forms.
                if old_g and any(_greek_key(g) not in catss_all_greek_keys
                                 for g in old_g):
                    fi += 1
                    continue
                if not _greek_lists_equal(old_g, new_g):
                    result['replaces'].append((ji, old_g, new_g))
                    hebrew_words[ji]['greek'] = new_g
                fi += 1

            if(SYNC_LEFTOVERS_FROM_CATSS_INTO_JSON):
                adds, removes, new_lo = reconcile_leftover(
                    leftover_greek, catss_leftover, hebrew_words, json_full_pool,
                    catss_all_greek_keys)
                result['leftover_adds']    = adds
                result['leftover_removes'] = removes
            if result['replaces'] or (SYNC_LEFTOVERS_FROM_CATSS_INTO_JSON and (adds or removes)):
                result['new_hebrew_words'] = hebrew_words
                result['new_leftover'] = new_lo if SYNC_LEFTOVERS_FROM_CATSS_INTO_JSON else leftover_greek
                result['status'] = 'exact'
            else:
                result['status'] = 'no_change'
            return result

    # ---------- Strategy 2: per-unique-word safe fixes ----------
    catss_by_key = defaultdict(list)
    for i, fl in enumerate(flat):
        keys = set()
        ka = strip_hebrew_for_match(fl['heb'])
        if ka:
            keys.add(ka)
        for alt in fl.get('heb_alt', []):
            kb = strip_hebrew_for_match(alt)
            if kb:
                keys.add(kb)
        for k in keys:
            catss_by_key[k].append((i, fl['greek']))

    json_by_key = defaultdict(list)
    for i, hw in enumerate(hebrew_words):
        k = strip_hebrew_for_match(hw.get('hebrew',''))
        if k:
            json_by_key[k].append(i)

    any_fix = False
    for k, catss_hits in catss_by_key.items():
        json_hits = json_by_key.get(k, [])
        if not json_hits:
            continue
        cc = len(catss_hits)
        jc = len(json_hits)

        if cc == 1 and jc == 1:
            ji = json_hits[0]
            proposed = _filter_to_pool(list(catss_hits[0][1]))
            if not proposed and catss_hits[0][1]:
                continue
            new_g = _merge_greek_preserve_case(proposed, json_full_pool)
            old_g = list(hebrew_words[ji].get('greek', []))
            if not new_g and old_g:
                continue
            # Morphology protection: don't swap forms CATSS doesn't know
            if old_g and any(_greek_key(g) not in catss_all_greek_keys
                             for g in old_g):
                continue
            if not _greek_lists_equal(old_g, new_g):
                result['replaces'].append((ji, old_g, new_g))
                hebrew_words[ji]['greek'] = new_g
                any_fix = True
        elif cc == jc and cc >= 2:
            first = catss_hits[0][1]
            all_same = all(_greek_lists_equal(first, h[1]) for h in catss_hits)
            if all_same:
                proposed = _filter_to_pool(list(first))
                if not proposed and first:
                    continue
                new_g = _merge_greek_preserve_case(proposed, json_full_pool)
                for ji in json_hits:
                    old_g = list(hebrew_words[ji].get('greek', []))
                    if not new_g and old_g:
                        continue
                    if old_g and any(_greek_key(g) not in catss_all_greek_keys
                                     for g in old_g):
                        continue
                    if not _greek_lists_equal(old_g, new_g):
                        result['replaces'].append((ji, old_g, new_g))
                        hebrew_words[ji]['greek'] = new_g
                        any_fix = True
            # else: positionally ambiguous, skip

    if(SYNC_LEFTOVERS_FROM_CATSS_INTO_JSON):
        adds, removes, new_lo = reconcile_leftover(
            leftover_greek, catss_leftover, hebrew_words, json_full_pool,
            catss_all_greek_keys)
        result['leftover_adds']    = adds
        result['leftover_removes'] = removes

    if any_fix or (SYNC_LEFTOVERS_FROM_CATSS_INTO_JSON and (adds or removes)):
        result['new_hebrew_words'] = hebrew_words
        result['new_leftover']     = new_lo if SYNC_LEFTOVERS_FROM_CATSS_INTO_JSON else leftover_greek
        result['status'] = 'partial'
    else:
        result['status'] = 'length_mismatch_skip' if len(flat) != len(hebrew_words) else 'no_change'
    return result


def reconcile_leftover(current_leftover, catss_leftover, hebrew_words, json_full_pool, catss_all_greek_keys):
    '''
    Leftover reconciliation, case-insensitive:
      - Any CATSS leftover greek word whose normalized key isn't already in
        the JSON's full greek pool → added (using preserved case if we've
        seen that key anywhere in the verse).
      - Any JSON leftover greek word whose normalized key matches a mapped
        word in hebrew_words AND isn't in catss_leftover → removed as dup.
      - If CATSS_AUTHORITATIVE_LEFTOVERS is set, any JSON leftover greek
        word whose normalized key doesn't appear in CATSS's full content for
        this verse (mapped + leftover) → removed.  This cleans up residual
        "wrong-verse leftovers" the AI pipeline created in books with LXX/MT
        verse-numbering shifts.
    '''
    mapped_keys = Counter()
    for hw in hebrew_words:
        for g in hw.get('greek', []):
            mapped_keys[_greek_key(g)] += 1
    full_keys = Counter(mapped_keys)
    for g in current_leftover:
        full_keys[_greek_key(g)] += 1

    catss_left_keyed = Counter(_greek_key(w) for w in catss_leftover)

    adds = []
    for w in catss_leftover:
        k = _greek_key(w)
        have = full_keys.get(k, 0)
        want = catss_left_keyed.get(k, 0)
        if have < want:
            preserved = None
            for jw in json_full_pool:
                if _greek_key(jw) == k:
                    preserved = jw
                    break
            adds.append(preserved if preserved is not None else w)
            full_keys[k] = have + 1

    new_leftover = list(current_leftover)
    removes = []
    # If CATSS has *no* Greek content for this verse at all, it cannot
    # authoritatively say which leftover words are "wrong" — disable the
    # authoritative-removal heuristic entirely for that case.
    catss_knows_anything = bool(catss_all_greek_keys)

    for w in list(new_leftover):
        k = _greek_key(w)
        if mapped_keys.get(k, 0) >= 1 and catss_left_keyed.get(k, 0) == 0:
            new_leftover.remove(w)
            removes.append(w)
            continue
        if (CATSS_AUTHORITATIVE_LEFTOVERS and catss_knows_anything
                and k not in catss_all_greek_keys):
            new_leftover.remove(w)
            removes.append(w)

    new_leftover.extend(adds)
    return adds, removes, new_leftover


## Serializer (same compact format as 4correct_json.ipynb)

In [ ]:
def _word_to_compact(w):
    heb     = w.get('hebrew',  '')
    greek   = w.get('greek',   [])
    latvian = w.get('latvian', [])
    hb_str = json.dumps(heb,     ensure_ascii=False)
    gr_str = json.dumps(greek,   ensure_ascii=False)
    lv_str = json.dumps(latvian, ensure_ascii=False)
    return f'{{ "hebrew": {hb_str}, "greek": {gr_str}, "latvian": {lv_str} }}'

def write_condensed_json(path, data):
    verse_blocks = []
    for verse in data:
        words = verse.get('hebrew_words', [])
        compact_words = ',\n      '.join(_word_to_compact(w) for w in words)
        lo_lv = json.dumps(verse.get('leftover_latvian', []), ensure_ascii=False)
        lo_gr = json.dumps(verse.get('leftover_greek',   []), ensure_ascii=False)
        verse_blocks.append(
            f'  {{\n    "hebrew_words": [\n      {compact_words}\n    ],\n'
            f'    "leftover_greek": {lo_gr},\n'
            f'    "leftover_latvian": {lo_lv}\n  }}'
        )
    new_raw = '[\n' + ',\n'.join(verse_blocks) + '\n]'
    with open(path, 'w', encoding='utf-8') as f:
        f.write(new_raw)


## Chapter-level driver

In [ ]:
GREEN = '\033[92m'; RED = '\x1b[38;5;196m'; BLUE = '\x1b[38;5;33m'; DIM = '\x1b[2m'; RESET = '\033[0m'

def process_chapter(bible_book, chapter_num, catss_verses_by_stem, stats, apply_changes=False):
    '''
    catss_verses_by_stem : dict {stem: {(chap, verse): cells}}
      Multiple stems = multiple CATSS traditions for this book (e.g. JoshA+JoshB).
      We apply them in order — second pass only makes safe fixes on top of first.
    '''
    src_path = Path(BIBLE_DIR) / bible_book / f'{chapter_num}.json'
    if not src_path.exists():
        return
    with open(src_path, 'r', encoding='utf-8') as f:
        verses_list = json.load(f)

    any_file_change = False
    for vi, verse_data in enumerate(verses_list):
        verse_num = vi + 1
        # translate MT (json) verse key → CATSS verse key
        remap, rstatus = catss_key_for_json(bible_book, chapter_num, verse_num)
        if rstatus == 'skip_merge':
            stats['status_skip_merge'] += 1
            if verbose_skipped:
                print(f'  {DIM}⏭  {bible_book} {chapter_num}:{verse_num} — merge/split group, not touched{RESET}')
            continue
        if rstatus == 'skip_missing':
            stats['status_skip_missing'] += 1
            continue
        catss_key = (remap[1], remap[2])  # (chap, verse) for catss lookup
        key = catss_key  # for printing consistency

        # accumulate fixes across CATSS traditions
        for stem, catss_map in catss_verses_by_stem.items():
            #catss_map = remapp(catss_map, bible_book)
            cells = catss_map.get(catss_key)
            if cells is None:
                continue
            diff = verse_catss_diff(verse_data, cells, (bible_book, chapter_num, verse_num), verbose=verbose)
            stats[f'status_{diff["status"]}'] += 1

            if diff['new_hebrew_words'] is not None:
                verse_data['hebrew_words']  = diff['new_hebrew_words']
                verse_data['leftover_greek'] = diff['new_leftover']
                verses_list[vi] = verse_data
                any_file_change = True

                stats['replaces']         += len(diff['replaces'])
                stats['leftover_adds']    += len(diff['leftover_adds'])
                stats['leftover_removes'] += len(diff['leftover_removes'])

                if verbose and (diff['replaces'] or diff['leftover_adds'] or diff['leftover_removes']):
                    remap_note = f' (←catss {catss_key[0]}:{catss_key[1]})' if rstatus == 'remapped' else ''
                    print(f'  {BLUE}[{stem}]{RESET} {bible_book} {chapter_num}:{verse_num}{remap_note} '
                          f'{diff["status"]} — {len(diff["replaces"])} replace, '
                          f'+{len(diff["leftover_adds"])}/-{len(diff["leftover_removes"])} leftovers')
                    for wi, old, new in diff['replaces'][:6]:
                        h = verse_data['hebrew_words'][wi].get('hebrew','')
                        print(f'    [{wi+1}] {h}  {RED}{old}{RESET} → {GREEN}{new}{RESET}')
                    if len(diff['replaces']) > 6:
                        print(f'    … and {len(diff["replaces"])-6} more')
                    if diff['leftover_adds']:
                        print(f'    {GREEN}+lo{RESET} {diff["leftover_adds"]}')
                    if diff['leftover_removes']:
                        print(f'    {RED}-lo{RESET} {diff["leftover_removes"]}')
            elif diff['status'] == 'length_mismatch_skip' and verbose_skipped:
                flat, _ = flatten_cells(cells)
                remap_note = f' (←catss {catss_key[0]}:{catss_key[1]})' if rstatus == 'remapped' else ''
                print(f'  {DIM}⏭  [{stem}]{RESET} {bible_book} {chapter_num}:{verse_num}{remap_note} '
                      f'length mismatch (catss={len(flat)} json={len(verse_data.get("hebrew_words",[]))}) '
                      f'— no safe word-level fix either')

    if apply_changes and any_file_change:
        write_condensed_json(src_path, verses_list)


## CATSS cache loader

In [ ]:
# Load each .par file once, keyed by stem. The new raw-.par parser
# (parse_par defined in cell above) replaces parse_par_html.
_catss_cache = {}
def load_catss_for_bible(bible_book):
    stems = BIBLE_TO_CATSS.get(bible_book, [])
    result = {}
    for stem in stems:
        if stem not in _catss_cache:
            # CCAT raw .par filenames use underscores in some books
            # (e.g. 10_Ruth.par) and dots in others (e.g. 20.Psalms.par
            # or 01.Genesis.par). Try both.
            candidates = [
                PAR_DIR_P / f'{stem}.par',
                PAR_DIR_P / f'{stem.replace(".", "_")}.par',
            ]
            fp = next((c for c in candidates if c.exists()), None)
            if fp is None:
                print(f'⚠️  no .par file found for {stem} (tried {[str(c) for c in candidates]}), skipping')
                _catss_cache[stem] = {}
            else:
                _catss_cache[stem] = parse_par(fp)
        result[stem] = _catss_cache[stem]
    return result


# Dry run on a single book (sanity)

In [ ]:
# Flip to a real book name to sanity-check before the full loop
from collections import Counter as _C
_test_stats = _C()
_test_book = 'numbers'
_test_catss = load_catss_for_bible(_test_book)
print(f'{_test_book}: loaded {len(_test_catss)} CATSS tradition(s): {list(_test_catss.keys())}')

_test_book_path = Path(BIBLE_DIR) / _test_book
if _test_book_path.exists():
    _chapters = sorted(int(f.stem) for f in _test_book_path.glob('*.json') if f.stem.isdigit())
    # do just chapter 1 for the dry run
    for ch in _chapters[:1]:
        process_chapter(_test_book, ch, _test_catss, _test_stats, apply_changes=False)
    print()
    print('=== dry-run stats (chapter 1 only, report-only) ===')
    for k, v in sorted(_test_stats.items()):
        print(f'  {k}: {v}')
else:
    print(f'{_test_book_path} does not exist yet — skip dry run')


# Full run — all books

In [ ]:
from collections import Counter
stats = Counter()
APPLY_CATSS_CHANGES=1
verbose=0
books = sorted([d.name for d in Path(BIBLE_DIR).iterdir()
                if d.is_dir() and d.name != 'excel'])# and d.name=='psalms'])# and d.name[0]<='d' ])#and d.name=='1_chronicles'])

for book in tqdm(books, desc='🪛 CATSS-fix'):
    if book not in BIBLE_TO_CATSS:
        print(f'⏭  {book}: no CATSS mapping, skipping')
        continue
    catss = load_catss_for_bible(book)
    if not any(catss.values()):
        continue
    for stem, catss_map in catss.items():
        catss_map = remapp(catss_map, book)
        catss[stem] = catss_map
        #print(f'  {book}: CATSS tradition {stem} has {len(catss_map)} verses')
    book_path = Path(BIBLE_DIR) / book
    chapters = sorted(int(f.stem) for f in book_path.glob('*.json') if f.stem.isdigit())
    if not chapters:
        continue
    for ch in tqdm(chapters, desc=f'📖 {book}', leave=False):
        process_chapter(book, ch, catss, stats, apply_changes=bool(APPLY_CATSS_CHANGES))

print()
print('=' * 60)
print(f'{"APPLIED" if APPLY_CATSS_CHANGES else "REPORT ONLY"} — summary')
print('=' * 60)
for k in sorted(stats):
    print(f'  {k:30s}: {stats[k]}')


In [ ]:
#os.chdir('..')
# os.chdir('bib_local')
print(os.getcwd())

## Notes on safety

- **Report mode (`APPLY_CATSS_CHANGES=0`)** never touches disk. Use this
  first to see what would change, fix any surprises, then re-run with `=1`.
- **`CATSS_AUTHORITATIVE_LEFTOVERS=1`** (default) drops any greek word from
  a verse's `leftover_greek` if CATSS doesn't know about that word for that
  verse. This cleans up the residual "wrong-verse leftovers" the prior AI
  pipeline created in books with LXX/MT numbering shifts (1 Samuel 24,
  Numbers 17, Daniel 3-4, Psalms titles, etc.). Set to `0` if you want to
  preserve pre-existing leftovers you've manually curated.
- **Verse-number remapping** uses a verbatim copy of the `l65_to_hb` table
  from `4correct_json.ipynb` cell 7. The Latvian 1965 Bible uses the same
  English verse numbering that CATSS uses, so the Hebrew↔Latvian map also
  works as Hebrew↔CATSS. Merge/split verses (marked with `!` in the
  derivation) are skipped with status `skip_merge` — too ambiguous to
  reassign Greek safely.
- **CATSS UI annotation icons (`ⓘ`)** are stripped before any text
  extraction. The new HTML format wraps every CATSS metadata note
  (`{d}`, `{t}`, `{...}`, `{..d}`, etc.) in `<span class="anno"
  title="..." data-anno="...">ⓘ</span>` for human readers; the parser
  removes these spans so the glyph never bleeds into the data.
- **Question-mark uncertainty markers (`?`, `??`)** per CATSS spec
  ("Questionable notation, equivalent, etc.") are stripped from
  tokenized Hebrew and Greek lists. They are not real words.
- **Questioned-minus variants** (`---?`, `--?`, `--- ?`) are recognized
  as plain minus rows and handled by the same minus protection rule:
  the engine will not wipe a non-empty JSON greek with an empty CATSS
  minus, regardless of whether it's questioned or definite.
- **Minus detection works for both old and new HTML formats.** The old
  4-column files marked minus rows with `class="grk-native minus"`; the
  new 5-column files don't set the CSS class but use `---` text content.
  The parser checks both, so both formats work transparently.
- **HTML parsing uses BeautifulSoup with class-aware lookup**, robust to:
  - the original 4-column layout (`heb-native, heb-trans, grk-native, grk-trans`)
  - the newer 5-column layout that wedges a `colb` column between heb-trans
    and grk-native (containing column-b retroversion text + nested HTML
    markup like `<br>`, `<span style=…>`, `<bdi class="ref">⟨ge10.4⟩</bdi>`)
  - `<tbody>` wrappers, missing trailing space in class attributes,
    `&zwj;` zero-width joiners, and inner tags inside data cells
  Columns are identified by their CSS class name, not by positional index,
  so additional columns won't shift the meaning of any cell.
- **Column-b (LXX-parent retroversion) fallback for Hebrew matching.**
  The new HTML format exposes CATSS column-b — the Hebrew the LXX-parent
  text seems to reflect, often differing from MT (e.g. column-A `ודומה`
  vs column-B `ידומה`). When the column-A Hebrew of a CATSS cell does
  not match the JSON's MT Hebrew, the engine falls back to column-B's
  Hebrew before giving up. This applies to both strategy 1 (positional
  exact-match) and strategy 2 (per-unique-word safe fixes).
- **CATSS minus protection.** When CATSS asserts that a Hebrew word has
  no Greek counterpart (a `---` minus, often part of a long minus block
  marked with `''` in the source `.par`), the engine **never wipes a
  non-empty JSON greek value to `[]`**. CATSS minuses often disagree with
  Rahlfs / Göttingen, which is what the JSON's AI mappings are based on,
  so the JSON's Greek is more authoritative for those cases. Verses where
  *every* CATSS row is a minus get status `catss_long_minus` and are
  skipped entirely. This protects e.g. 1 Chronicles 1:11–24 (the Table of
  Nations names that CATSS marks as minus but Rahlfs has).
- **CATSS-authoritative leftover removal is also disabled** for verses
  where CATSS knows nothing about the verse's Greek content (no mapped
  greek and no leftover greek). It can only remove leftover words when
  CATSS positively asserts something about the verse.
- **Multi-word Hebrew cells preserve existing JSON splits.** When CATSS
  groups N Hebrew words into one alignment cell with a single Greek group
  (e.g. `את נמרוד → τὸν νεβρωδ`), the engine first checks whether the
  JSON's existing per-word distribution across those N words concatenates
  to the same Greek (case-insensitively). If yes, the JSON is left alone
  — its split is at least as informative as the lump. Only when the
  contents differ do we replace, putting the whole Greek group on the
  first Hebrew word.
- **CATSS spec markers handled** per `00.ReadReParallel.txt`:
  `''` (long minus/plus marker) rows are skipped entirely, neither parsed
  as Hebrew nor as Greek words. Any prior-run corruption where `"''"`
  was written into JSON greek lists is auto-cleaned on the next pass.
  `~~~` / `^^^` transposition pointers are resolved **bidirectionally** —
  the moved Greek can appear either before or after its anchor Hebrew word
  in the verse (e.g. 1 Chronicles 4:26 has the `αμουηλ` pointer at row 1
  while its anchor `XMW)L` is at row 4).
- **Multiple CATSS traditions for one bible book** (JoshA+B, JudgesA+B,
  DanielOG+Th) are applied in order. Because the engine only makes
  strictly-safe fixes on length mismatches, a second-tradition pass can
  only correct additional words that the first pass left alone.
- **Hebrew comparison is loose** (consonants only, final-form collapsed,
  slashes removed). This handles pointed MT ↔ unpointed CATSS as well as
  ם/מ etc.
- **Greek comparison is case-and-accent-insensitive**, and when a
  replacement is needed we preserve the JSON's original capitalization
  where possible (so `Ααρων`, `Λευῖται`, `Ισραηλ` stay capitalized even
  though CATSS stores them lowercase).
- **This notebook never edits Hebrew or Latvian fields** — run
  `4correct_json.ipynb` first to get those right, then this one.
